# Problema 09: Espectro de Diseno Eurocode 8

## Relacion 5 - Ingenieria Sismica

**Objetivo:** Construir el espectro elastico de respuesta segun Eurocode 8 para diferentes tipos de suelo y analizar sus caracteristicas.

### Enunciado

Se desea obtener el espectro de diseno para una zona sismica con:
- Aceleracion de referencia: $a_{gR} = 0.16g$
- Coeficiente de importancia: $\gamma_I = 1.2$

Se pide:
1. Calcular el espectro elastico para suelos tipo A, B, C y D
2. Identificar las zonas caracteristicas del espectro
3. Comparar los espectros para diferentes tipos de suelo

## Datos del Problema

| Parametro | Valor | Descripcion |
|-----------|-------|-------------|
| $a_{gR}$ | 0.16g | Aceleracion de referencia |
| $\gamma_I$ | 1.2 | Coeficiente de importancia |
| $a_g$ | 0.192g | Aceleracion de diseno ($a_g = \gamma_I \cdot a_{gR}$) |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Datos del problema
agR = 0.16  # g - Aceleracion de referencia
gamma_I = 1.2  # Coeficiente de importancia
ag = gamma_I * agR  # Aceleracion de diseno (en unidades de g)
g = 9.81  # m/s^2

print("DATOS DEL PROBLEMA")
print("="*40)
print(f"Aceleracion de referencia: agR = {agR}g")
print(f"Coeficiente de importancia: gamma_I = {gamma_I}")
print(f"Aceleracion de diseno: ag = {ag:.3f}g = {ag*g:.2f} m/s^2")

## 1. Parametros por Tipo de Suelo (EC8)

Eurocode 8 define 5 tipos de suelo con los siguientes parametros para el espectro Tipo 1:

| Suelo | S | $T_B$ (s) | $T_C$ (s) | $T_D$ (s) | Descripcion |
|-------|---|-----------|-----------|-----------|-------------|
| A | 1.0 | 0.15 | 0.4 | 2.0 | Roca |
| B | 1.2 | 0.15 | 0.5 | 2.0 | Suelo muy rigido |
| C | 1.15 | 0.20 | 0.6 | 2.0 | Suelo medianamente denso |
| D | 1.35 | 0.20 | 0.8 | 2.0 | Suelo blando |

In [ ]:
# Parametros por tipo de suelo (EC8 Tipo 1)
suelos = {
    'A': {'S': 1.0, 'TB': 0.15, 'TC': 0.4, 'TD': 2.0, 'color': 'blue', 
          'desc': 'Roca o formacion similar'},
    'B': {'S': 1.2, 'TB': 0.15, 'TC': 0.5, 'TD': 2.0, 'color': 'green', 
          'desc': 'Depositos de arena muy densa'},
    'C': {'S': 1.15, 'TB': 0.20, 'TC': 0.6, 'TD': 2.0, 'color': 'orange', 
          'desc': 'Arena densa a medianamente densa'},
    'D': {'S': 1.35, 'TB': 0.20, 'TC': 0.8, 'TD': 2.0, 'color': 'red', 
          'desc': 'Depositos de suelo blando'}
}

print("PARAMETROS EC8 POR TIPO DE SUELO")
print("="*65)
print(f"{'Suelo':<8} {'S':<8} {'TB (s)':<10} {'TC (s)':<10} {'TD (s)':<10}")
print("-"*65)
for tipo, params in suelos.items():
    print(f"{tipo:<8} {params['S']:<8.2f} {params['TB']:<10.2f} {params['TC']:<10.2f} {params['TD']:<10.2f}")
    print(f"         {params['desc']}")

## 2. Definicion del Espectro Elastico EC8

El espectro elastico horizontal de aceleraciones se define por zonas:

**Zona 1: $0 \le T \le T_B$ (Rama ascendente)**
$$S_e(T) = a_g \cdot S \cdot \left[1 + \frac{T}{T_B}(\eta \cdot 2.5 - 1)\right]$$

**Zona 2: $T_B < T \le T_C$ (Plateau)**
$$S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5$$

**Zona 3: $T_C < T \le T_D$ (Descenso proporcional a $1/T$)**
$$S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5 \cdot \frac{T_C}{T}$$

**Zona 4: $T > T_D$ (Descenso proporcional a $1/T^2$)**
$$S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5 \cdot \frac{T_C \cdot T_D}{T^2}$$

Donde:
- $\eta = \sqrt{10/(5+\zeta)} \ge 0.55$ es el factor de correccion por amortiguamiento
- Para $\zeta = 5\%$: $\eta = 1.0$

In [ ]:
def Se_EC8(T, ag, S, TB, TC, TD, eta=1.0):
    """
    Calcula el espectro elastico EC8.
    
    Parametros:
        T: periodo(s) - puede ser array
        ag: aceleracion de diseno (en unidades de g)
        S: factor de amplificacion del suelo
        TB: limite inferior del plateau
        TC: limite superior del plateau
        TD: inicio de la zona de desplazamiento constante
        eta: factor de correccion por amortiguamiento (1.0 para 5%)
    
    Retorna:
        Se: aceleracion espectral (en unidades de g)
    """
    T = np.atleast_1d(T)
    Se = np.zeros_like(T)
    
    for i, t in enumerate(T):
        if t <= 0:
            Se[i] = ag * S
        elif t <= TB:
            # Zona 1: Rama ascendente
            Se[i] = ag * S * (1 + t/TB * (eta*2.5 - 1))
        elif t <= TC:
            # Zona 2: Plateau
            Se[i] = ag * S * eta * 2.5
        elif t <= TD:
            # Zona 3: Descenso 1/T
            Se[i] = ag * S * eta * 2.5 * TC / t
        else:
            # Zona 4: Descenso 1/T^2
            Se[i] = ag * S * eta * 2.5 * TC * TD / t**2
    
    return Se

# Rango de periodos
T = np.linspace(0.01, 4.0, 500)

print("FUNCION Se_EC8 DEFINIDA")
print("="*40)
print("La funcion calcula el espectro elastico")
print("segun las formulas del Eurocode 8.")

## 3. Calculo de Espectros por Tipo de Suelo

In [ ]:
# Calcular espectros
espectros = {}
for tipo, params in suelos.items():
    espectros[tipo] = Se_EC8(T, ag, params['S'], params['TB'], params['TC'], params['TD'])

print("VALORES MAXIMOS DEL ESPECTRO (PLATEAU)")
print("="*50)
print(f"{'Suelo':<8} {'S':<6} {'Se_max (g)':<12} {'Se_max (m/s^2)':<15}")
print("-"*50)
for tipo, params in suelos.items():
    Se_max = ag * params['S'] * 2.5
    print(f"{tipo:<8} {params['S']:<6.2f} {Se_max:<12.3f} {Se_max*g:<15.2f}")

## 4. Visualizacion de Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Espectros por tipo de suelo
ax1 = axes[0]
for tipo, params in suelos.items():
    ax1.plot(T, espectros[tipo], params['color'], linewidth=2,
             label=f"Suelo {tipo} ({params['desc'][:15]}...)")

ax1.axhline(y=ag, color='k', linestyle='--', alpha=0.5, label=f'ag = {ag:.3f}g')
ax1.set_xlabel('Periodo T (s)', fontsize=12)
ax1.set_ylabel('Se/g', fontsize=12)
ax1.set_title('Espectro elastico EC8 por tipo de suelo', fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 4)
ax1.set_ylim(0, 0.7)

# Panel 2: Espectro suelo C con zonas marcadas
ax2 = axes[1]
params_C = suelos['C']
Se_C = espectros['C']
ax2.plot(T, Se_C, 'orange', linewidth=2.5, label='Suelo C')

# Marcar puntos caracteristicos
TB, TC, TD = params_C['TB'], params_C['TC'], params_C['TD']
ax2.axvline(x=TB, color='gray', linestyle=':', alpha=0.7)
ax2.axvline(x=TC, color='gray', linestyle=':', alpha=0.7)
ax2.axvline(x=TD, color='gray', linestyle=':', alpha=0.7)

ax2.annotate(f'TB={TB}s', (TB, 0.02), fontsize=10, ha='center')
ax2.annotate(f'TC={TC}s', (TC, 0.02), fontsize=10, ha='center')
ax2.annotate(f'TD={TD}s', (TD, 0.02), fontsize=10, ha='center')

# Rellenar zonas
ax2.fill_between(T[T<=TB], 0, Se_C[T<=TB], alpha=0.2, color='blue', label='Rampa')
ax2.fill_between(T[(T>TB)&(T<=TC)], 0, Se_C[(T>TB)&(T<=TC)], alpha=0.2, color='green', label='Plateau')
ax2.fill_between(T[(T>TC)&(T<=TD)], 0, Se_C[(T>TC)&(T<=TD)], alpha=0.2, color='orange', label='Descenso 1/T')
ax2.fill_between(T[T>TD], 0, Se_C[T>TD], alpha=0.2, color='red', label='Descenso 1/T^2')

ax2.set_xlabel('Periodo T (s)', fontsize=12)
ax2.set_ylabel('Se/g', fontsize=12)
ax2.set_title('Zonas del espectro EC8 (Suelo C)', fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 0.7)

plt.tight_layout()
plt.show()

## 5. Analisis de la Influencia del Tipo de Suelo

In [ ]:
# Comparar valores para diferentes periodos tipicos
periodos_tipicos = [0.1, 0.3, 0.5, 1.0, 2.0, 3.0]

print("VALORES ESPECTRALES PARA PERIODOS TIPICOS")
print("="*70)
print(f"{'T (s)':<10}", end="")
for tipo in suelos.keys():
    print(f"{'Suelo '+tipo:<12}", end="")
print(f"{'Max/Min':<12}")
print("-"*70)

for T_val in periodos_tipicos:
    valores = []
    print(f"{T_val:<10.1f}", end="")
    for tipo, params in suelos.items():
        Se_val = Se_EC8(T_val, ag, params['S'], params['TB'], params['TC'], params['TD'])[0]
        valores.append(Se_val)
        print(f"{Se_val:<12.3f}", end="")
    ratio = max(valores) / min(valores)
    print(f"{ratio:<12.2f}")

In [ ]:
# Grafico de ratios respecto a suelo A
fig, ax = plt.subplots(figsize=(10, 5))

Se_A = espectros['A']
for tipo in ['B', 'C', 'D']:
    ratio = espectros[tipo] / Se_A
    ax.plot(T, ratio, suelos[tipo]['color'], linewidth=2, label=f'Suelo {tipo} / Suelo A')

ax.axhline(y=1, color='k', linestyle='--', linewidth=0.5)
ax.set_xlabel('Periodo T (s)', fontsize=12)
ax.set_ylabel('Ratio Se/Se_A', fontsize=12)
ax.set_title('Amplificacion respecto a suelo tipo A (roca)', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4)
ax.set_ylim(0.5, 2.5)

plt.tight_layout()
plt.show()

print("\nOBSERVACION: Los suelos blandos amplifican las aceleraciones,")
print("especialmente para periodos largos (estructuras flexibles).")

## 6. Efecto del Amortiguamiento

In [ ]:
def eta_factor(zeta):
    """Factor de correccion por amortiguamiento (EC8)."""
    eta = np.sqrt(10 / (5 + zeta*100))
    return max(eta, 0.55)

# Calcular espectros para diferentes amortiguamientos
zetas = [0.02, 0.05, 0.10, 0.20]
params_C = suelos['C']

fig, ax = plt.subplots(figsize=(10, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(zetas)))
for i, zeta in enumerate(zetas):
    eta = eta_factor(zeta)
    Se = Se_EC8(T, ag, params_C['S'], params_C['TB'], params_C['TC'], params_C['TD'], eta)
    ax.plot(T, Se, color=colors[i], linewidth=2, 
            label=f'zeta = {zeta*100:.0f}% (eta = {eta:.2f})')

ax.set_xlabel('Periodo T (s)', fontsize=12)
ax.set_ylabel('Se/g', fontsize=12)
ax.set_title('Efecto del amortiguamiento en el espectro (Suelo C)', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4)

plt.tight_layout()
plt.show()

print("FACTORES DE CORRECCION POR AMORTIGUAMIENTO")
print("="*40)
for zeta in zetas:
    eta = eta_factor(zeta)
    reduccion = (1 - eta) * 100
    print(f"zeta = {zeta*100:>4.0f}%: eta = {eta:.3f} (reduccion = {reduccion:+.1f}%)")

## Interpretacion de Resultados

### Zonas del espectro
1. **Zona de rampa** ($0 < T < T_B$): La aceleracion espectral crece linealmente desde $a_g \cdot S$ hasta el maximo.
2. **Zona plateau** ($T_B < T < T_C$): Region de maxima amplificacion ($S_e = a_g \cdot S \cdot 2.5$). Corresponde a estructuras rigidas.
3. **Zona de velocidad constante** ($T_C < T < T_D$): La aceleracion decrece como $1/T$. Estructuras de rigidez media.
4. **Zona de desplazamiento constante** ($T > T_D$): La aceleracion decrece como $1/T^2$. Estructuras muy flexibles.

### Efecto del tipo de suelo
- Los suelos blandos (D) amplifican mas las aceleraciones que los suelos duros (A).
- El efecto de sitio desplaza el plateau hacia periodos mayores.
- Estructuras flexibles sobre suelos blandos son especialmente vulnerables.

### Efecto del amortiguamiento
- Mayor amortiguamiento reduce las aceleraciones espectrales.
- El factor eta penaliza estructuras con bajo amortiguamiento.
- Para zeta > 10%, la reduccion es significativa.